# Battle Lab · CENSUS_EXTENDED M-C 🐉📊

Extensión **incremental y BO3-only** del censo humano de `BATTLE-LAB-MC-TRAIN-001`.

Reutiliza los 5,000 logs BO3 ya persistidos, busca hacia atrás hasta un objetivo de **10,000 logs BO3 aceptados**, reconstruye trayectorias con todo el corpus acumulado y vuelve a evaluar el gate BC-MC (**1,000 trayectorias / 10,000 transiciones**).

No reescanea el ladder M-C normal y no inicia entrenamiento.


In [ ]:
TARGET_BO3_LOGS = 10_000  # @param {type:"integer"}
MIN_BC_TRAJECTORIES = 1000
MIN_BC_TRANSITIONS = 10000
MIN_RATING = 1200
ONLY_WINNER = True
SEED = 260913
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "main"
VGC_BENCH_REPOSITORY = "https://github.com/cameronangliss/vgc-bench.git"
VGC_BENCH_COMMIT = "d79f9532947ac114dce1dda2456a590afcd375b2"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
DATA_ROOT = DRIVE_ROOT / "data"
TEAM_DIR = DATA_ROOT / "teams" / "vgcpastes-champions-mc"
SPLIT_ROOT = DATA_ROOT / "team-splits" / f"seed-{SEED}"
OUTPUT_ROOT = DRIVE_ROOT / "training"
PKMN_ROOT = Path("/content/pkmn")
VGC_BENCH_ROOT = Path("/content/vgc-bench")
for path in (DATA_ROOT, OUTPUT_ROOT, SPLIT_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print("📁", DRIVE_ROOT)


In [ ]:
import os, subprocess, sys, time

def run(command, *, cwd=None, label=None):
    if label:
        print(f"\n▶ {label}", flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

if not (PKMN_ROOT / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", PKMN_REPOSITORY, str(PKMN_ROOT)], label="Clonando pkmn")
run(["git", "fetch", "origin", PKMN_REF], cwd=PKMN_ROOT, label="Actualizando pkmn")
run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=PKMN_ROOT, label="Fijando pkmn")
pkmn_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PKMN_ROOT, text=True).strip()
print("pkmn SHA:", pkmn_sha)

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase1.txt")], label="Dependencias Battle Lab")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-mc-train.txt")], label="Dependencias VGC-Bench")

sys.path.insert(0, str(PKMN_ROOT))
from battle_lab.vgc_bench_battle import ensure_vgc_bench_checkout
ensure_vgc_bench_checkout(
    checkout=VGC_BENCH_ROOT,
    repository=VGC_BENCH_REPOSITORY,
    commit=VGC_BENCH_COMMIT,
)
print("VGC-Bench SHA:", VGC_BENCH_COMMIT)


In [ ]:
import json

run([
    sys.executable, "-m", "battle_lab.mc_census_extended",
    "--vgc-bench", str(VGC_BENCH_ROOT),
    "--data-root", str(DATA_ROOT),
    "--max-logs", str(TARGET_BO3_LOGS),
    "--num-workers", "16",
    "--read-increment", "5000",
], cwd=PKMN_ROOT, label="Extendiendo únicamente M-C BO3")

# Reconstruye trayectorias con normal + BO3 acumulados y el filtro vigente.
args = [
    sys.executable, "-m", "battle_lab.mc_training",
    "--vgc-bench", str(VGC_BENCH_ROOT),
    "--project-root", str(PKMN_ROOT),
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--team-dir", str(TEAM_DIR),
    "build-trajectories",
    "--num-workers", str(max(2, (os.cpu_count() or 2) // 2)),
]
if MIN_RATING > 0:
    args += ["--min-rating", str(MIN_RATING)]
if ONLY_WINNER:
    args += ["--only-winner"]
run(args, cwd=PKMN_ROOT, label="Reconstruyendo trayectorias filtradas")

run([
    sys.executable, "-m", "battle_lab.mc_census",
    "--vgc-bench", str(VGC_BENCH_ROOT),
    "--data-root", str(DATA_ROOT),
    "--team-dir", str(TEAM_DIR),
    "--output-root", str(OUTPUT_ROOT),
    "report",
], cwd=PKMN_ROOT, label="Generando reporte final")

# Deja preparado el holdout por firma de seis especies para LIGHT.
run([
    sys.executable, "-m", "battle_lab.mc_team_split",
    "--source-dir", str(TEAM_DIR),
    "--output-root", str(SPLIT_ROOT),
    "--holdout-fraction", "0.20",
    "--seed", str(SEED),
], cwd=PKMN_ROOT, label="Preparando train/holdout sin fuga")


In [ ]:
import json

summary = json.loads((OUTPUT_ROOT / "census.json").read_text())
logs_manifest = json.loads((DATA_ROOT / "logs_manifest.json").read_text())
split_manifest = json.loads((SPLIT_ROOT / "split_manifest.json").read_text())
bc_ready = (
    int(summary["trajectories"]) >= MIN_BC_TRAJECTORIES
    and int(summary["transitions"]) >= MIN_BC_TRANSITIONS
)

print("\n===== CENSUS_EXTENDED M-C =====")
print("Logs OTS totales:", f"{summary['humanLogs']:,}")
print("Trayectorias:", f"{summary['trajectories']:,}")
print("Transiciones:", f"{summary['transitions']:,}")
print("BC-MC habilitable:", bc_ready)
print("Train / holdout:", split_manifest["trainTeams"], "/", split_manifest["holdoutTeams"])
print("Fuga de firmas:", split_manifest["signatureLeakage"])
print("Extensión BO3:", logs_manifest.get("extension", {}))

# Publica un resumen nativo legible por Drive/ChatGPT/Gem.
from google.colab import auth
from googleapiclient.discovery import build
auth.authenticate_user()
drive_api = build("drive", "v3")
docs_api = build("docs", "v1")

def find_folder(parent_id, name):
    safe = name.replace("'", "\\'")
    q = (
        f"name='{safe}' and '{parent_id}' in parents and trashed=false "
        "and mimeType='application/vnd.google-apps.folder'"
    )
    rows = drive_api.files().list(q=q, fields="files(id,name)", pageSize=10).execute().get("files", [])
    if not rows:
        raise RuntimeError(f"No se encontró la carpeta canónica: {name}")
    return rows[0]["id"]

parent_id = "root"
for folder_name in ("Colabs", "LikeNoOneEverWas", "BattleLab", "MC-Training", "training"):
    parent_id = find_folder(parent_id, folder_name)

title = "Battle Lab M-C — latest run"
safe_title = title.replace("'", "\\'")
q = (
    f"name='{safe_title}' and '{parent_id}' in parents and trashed=false "
    "and mimeType='application/vnd.google-apps.document'"
)
rows = drive_api.files().list(q=q, fields="files(id,name)", pageSize=10).execute().get("files", [])
if rows:
    doc_id = rows[0]["id"]
    doc = docs_api.documents().get(documentId=doc_id).execute()
    end_index = doc["body"]["content"][-1]["endIndex"]
    requests = []
    if end_index > 2:
        requests.append({"deleteContentRange": {"range": {"startIndex": 1, "endIndex": end_index - 1}}})
else:
    doc_id = drive_api.files().create(
        body={"name": title, "mimeType": "application/vnd.google-apps.document", "parents": [parent_id]},
        fields="id",
    ).execute()["id"]
    requests = []

verdict = (
    "Gate alcanzado: el siguiente LIGHT debe ejecutar BC-MC y luego PPO/self-play."
    if bc_ready
    else "Gate no alcanzado tras la extensión: el siguiente LIGHT debe partir del baseline BC M-A/M-B y hacer PPO/self-play."
)
extension = logs_manifest.get("extension", {})
report = f"""Battle Lab M-C — CENSUS_EXTENDED

{verdict}

Logs OTS totales: {summary['humanLogs']:,}
Trayectorias filtradas: {summary['trajectories']:,}
Transiciones: {summary['transitions']:,}
Gate: {MIN_BC_TRAJECTORIES:,} / {MIN_BC_TRANSITIONS:,}
BC-MC habilitable: {bc_ready}

BO3 inicial: {extension.get('startingAcceptedLogs', 'n/a')}
BO3 objetivo: {extension.get('targetAcceptedLogs', 'n/a')}
BO3 final: {extension.get('endingAcceptedLogs', 'n/a')}
BO3 añadidos: {extension.get('addedAcceptedLogs', 'n/a')}
Fuente agotada: {extension.get('sourceExhausted', 'n/a')}

Train / holdout: {split_manifest['trainTeams']} / {split_manifest['holdoutTeams']}
Fuga de firmas: {split_manifest['signatureLeakage']}

pkmn SHA: {pkmn_sha}
Drive: {DRIVE_ROOT}
"""
requests.append({"insertText": {"location": {"index": 1}, "text": report}})
docs_api.documents().batchUpdate(documentId=doc_id, body={"requests": requests}).execute()
print("📄 Resumen nativo:", f"https://docs.google.com/document/d/{doc_id}/edit")
